In [1]:
import torch
import torch.nn as nn
import tiktoken

from src.rope import compute_rope_params, apply_rope

In [2]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Tamaño del vocabulario (BPE de GPT-2)
    "context_length": 1024, # Longitud máxima de contexto
    "emb_dim": 768,         # Dimensión de incrustación
    "n_heads": 12,          # Número de cabezas de atención
    "n_layers": 12,         # Número de bloques Transformer
    "drop_rate": 0.1,       # Tasa de dropout
    "qkv_bias": True       # Bias en proyecciones Q, K, V (False en GPT modernos)
}

##### Mecanismo de Multi-Head Attention causal (source: chapter 3)

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out debe ser divisible por num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        cos, sin = compute_rope_params(
            head_dim=self.head_dim,
            context_length=context_length
        )

        self.register_buffer("rope_cos", cos)
        self.register_buffer("rope_sin", sin)

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        
        # Máscara causal triangular superior registrada como buffer (no entrenable)
        self.register_buffer(
            "mask", 
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)      # (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # Proyectar y desdoblar la dimensión de cabezas:
        # (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transponer a (b, num_heads, num_tokens, head_dim) para paralelizar
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        queries = apply_rope(queries, self.rope_cos, self.rope_sin)
        keys = apply_rope(keys, self.rope_cos, self.rope_sin)

        # Producto punto escalado
        attn_scores = queries @ keys.transpose(2, 3)

        # Truncar máscara al tamaño de la secuencia actual y aplicar
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / (keys.shape[-1] ** 0.5), dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Combinar cabezas
        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)

        return context_vec

In [4]:
torch.manual_seed(123)

mha = MultiHeadAttention(
    d_in=768,
    d_out=768,
    context_length=1024,
    dropout=0.0,
    num_heads=12,
    qkv_bias=False
)

x = torch.randn(
    2,
    16,
    768
)

output = mha(x)

print("Input :", x.shape)
print("Output:", output.shape)

Input : torch.Size([2, 16, 768])
Output: torch.Size([2, 16, 768])


##### Capas de soporte (LayerNorm, GELU y FeedForward) (source: chapter 4)

In [6]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * 
            (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

##### Transformer (source: chapter 4)

In [7]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Rama de Atención con Shortcut / Residual
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        # Rama Feed-Forward con Shortcut / Residual
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x

##### Arquitectura del modelo (source: chapter 4)

In [8]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

##### Generación de texto simple (source: chapter 4)

In [9]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        # Truncar si la secuencia supera el contexto admitido
        idx_cond = idx[:, -context_size:]
        
        with torch.no_grad():
            logits = model(idx_cond)
            
        # Extraer únicamente los logits del último token
        logits = logits[:, -1, :]
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        
        # Concatenar el nuevo token a la secuencia
        idx = torch.cat((idx, idx_next), dim=1)
        
    return idx

##### Prueba de ejecución

In [10]:
# Fijar semilla para reproducibilidad
torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Instanciar modelo
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
model.eval()

# Contar parámetros
total_params = sum(p.numel() for p in model.parameters())
print(f"Dispositivo: {device}")
print(f"Total de parámetros instanciados: {total_params:,}")

# 2. Tokenizar un prompt inicial
tokenizer = tiktoken.get_encoding("gpt2")
prompt = "Hello, I am a language model,"
encoded_input = torch.tensor(tokenizer.encode(prompt), device=device).unsqueeze(0)

# 3. Probar la pasada hacia adelante (Forward Pass)
with torch.no_grad():
    logits = model(encoded_input)
print(f"Dimensiones de salida (Logits): {logits.shape}")  # [1, len_prompt, 50257]

# 4. Generar tokens (Nota: como los pesos son aleatorios, la salida será incoherente)
out_tokens = generate_text_simple(
    model=model, 
    idx=encoded_input, 
    max_new_tokens=10, 
    context_size=GPT_CONFIG_124M["context_length"]
)
generated_text = tokenizer.decode(out_tokens.squeeze(0).tolist())

print("\n--- Texto generado (pesos no entrenados) ---")
print(generated_text)

Dispositivo: cuda
Total de parámetros instanciados: 163,037,184
Dimensiones de salida (Logits): torch.Size([1, 8, 50257])

--- Texto generado (pesos no entrenados) ---
Hello, I am a language model, KILL rect Aly Destination Written excessleneckTile Geralddot


##### Carga de pesos desde OpenAI (source: chapter 5)

In [15]:
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
model.eval()

from transformers import GPT2LMHeadModel

# Descarga los pesos originales de GPT-2 (124M) desde los servidores de Hugging Face / OpenAI
print("Descargando pesos oficiales de OpenAI GPT-2...")
gpt_hf = GPT2LMHeadModel.from_pretrained("gpt2")
gpt_hf.eval()
print("Pesos descargados con éxito.")

/home/usuario/A4/PL/venv_PL/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Descargando pesos oficiales de OpenAI GPT-2...


KeyboardInterrupt: 

In [ ]:
def load_weights_into_gpt(model, gpt_hf):
    # 1. Incrustaciones de tokens y posiciones
    model.tok_emb.weight.data.copy_(gpt_hf.transformer.wte.weight.data)
    model.pos_emb.weight.data.copy_(gpt_hf.transformer.wpe.weight.data)

    # 2. Transferencia bloque a bloque de los 12 TransformerBlocks
    for i, block in enumerate(gpt_hf.transformer.h):
        # Normalización 1 (LayerNorm)
        model.trf_blocks[i].norm1.scale.data.copy_(block.ln_1.weight.data)
        model.trf_blocks[i].norm1.shift.data.copy_(block.ln_1.bias.data)

        # Atención Causal: c_attn contiene Q, K y V juntos (forma: 768 x 2304)
        # Se transpone a (2304, 768) y se divide en 3 bloques de 768
        w_qkv = block.attn.c_attn.weight.T
        b_qkv = block.attn.c_attn.bias
        w_q, w_k, w_v = torch.chunk(w_qkv, 3, dim=0)
        b_q, b_k, b_v = torch.chunk(b_qkv, 3, dim=0)

        model.trf_blocks[i].att.W_query.weight.data.copy_(w_q)
        model.trf_blocks[i].att.W_query.bias.data.copy_(b_q)
        model.trf_blocks[i].att.W_key.weight.data.copy_(w_k)
        model.trf_blocks[i].att.W_key.bias.data.copy_(b_k)
        model.trf_blocks[i].att.W_value.weight.data.copy_(w_v)
        model.trf_blocks[i].att.W_value.bias.data.copy_(b_v)

        # Proyección de salida de atención
        model.trf_blocks[i].att.out_proj.weight.data.copy_(block.attn.c_proj.weight.T)
        model.trf_blocks[i].att.out_proj.bias.data.copy_(block.attn.c_proj.bias)

        # Normalización 2 (LayerNorm)
        model.trf_blocks[i].norm2.scale.data.copy_(block.ln_2.weight.data)
        model.trf_blocks[i].norm2.shift.data.copy_(block.ln_2.bias.data)

        # Red Feed-Forward (MLP)
        model.trf_blocks[i].ff.layers[0].weight.data.copy_(block.mlp.c_fc.weight.T)
        model.trf_blocks[i].ff.layers[0].bias.data.copy_(block.mlp.c_fc.bias)
        model.trf_blocks[i].ff.layers[2].weight.data.copy_(block.mlp.c_proj.weight.T)
        model.trf_blocks[i].ff.layers[2].bias.data.copy_(block.mlp.c_proj.bias)

    # 3. Normalización final y cabeza de predicción de vocabulario
    model.final_norm.scale.data.copy_(gpt_hf.transformer.ln_f.weight.data)
    model.final_norm.shift.data.copy_(gpt_hf.transformer.ln_f.bias.data)
    model.out_head.weight.data.copy_(gpt_hf.lm_head.weight.data)

# Ejecutar la transferencia
load_weights_into_gpt(model, gpt_hf)
model.to(device)
print("¡Pesos oficiales transferidos correctamente a tu GPTModel!")

In [ ]:
torch.manual_seed(123)

prompt = "Every effort moves you"
encoded_input = torch.tensor(tokenizer.encode(prompt), device=device).unsqueeze(0)

# Generar 25 tokens
out_tokens = generate_text_simple(
    model=model,
    idx=encoded_input,
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("\n--- Texto generado con pesos preentrenados ---")
print(tokenizer.decode(out_tokens.squeeze(0).tolist()))